In [ ]:
from pyspark.sql import SparkSession

# Initialize SparkSession
spark = SparkSession.builder.appName("MySparkApp").getOrCreate()

# Read the text file
lines = spark.read.text("Airline-Full-Non-Ag-DFE-Sentiment.csv")

# Split lines into parts
parts = lines.rdd.map(lambda line: line.value.split(","))

# Filter for valid parts
validParts = parts.filter(lambda line: len(line) >= 22 and line[14] is not None and line[21] is not None)

# Filter for negative sentiments
negatives = validParts.filter(lambda tweet: tweet[14] == "negative")

# Extract tweet texts
tweetTexts = negatives.map(lambda tweet: tweet[21])

# Split tweet texts into words
words = tweetTexts.flatMap(lambda text: text.split(" "))

# Filter for valid words
validWords = words.filter(lambda word: word is not None and word != "")

# Modify and clean the words
keyValues = validWords.map(lambda word: (word.lower().replace("\"", "").replace("@", "").replace("aa", "americanair").replace("cancelled", "canceled")))

# Reduce by key to get word counts
wordCounts = keyValues.countByValue()

# Swap key-value pairs
countWords = [(v, k) for k, v in wordCounts.items()]

# Sort the word counts in descending order
sortedCounts = sorted(countWords, key=lambda x: x[0], reverse=True)

# Read most frequent words from a text file
frequentWords = spark.read.text("mostfrequentwords.txt").rdd.map(lambda line: line.value).collect()

# Convert the frequent words to a set for efficient filtering
fwSet = set(frequentWords)

# Filter out the frequent words
results = [x for x in sortedCounts if x[1] not in fwSet]

# Take the top 20 results and print them
for i, (count, word) in enumerate(results[:20], start=1):
    print(f"{i}: {word}: {count}")


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/03/19 23:10:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/03/19 23:10:06 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


1: americanair: 10146
2: united: 9144
3: usairways: 7270
4: southwestair: 4713
5: jetblue: 3861
6: customer: 2433
7: hold: 2307
8: canceled: 1931
9: hour: 1592
10: call: 1563
11: why: 1536
12: service: 1495
13: still: 1433
14: hours: 1294
15: 2: 1279
16: please: 1254
17: need: 1207
18: delayed: 1076
19: waiting: 1053
20: trying: 1042


25/03/20 05:19:38 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 586957 ms exceeds timeout 120000 ms
25/03/20 05:19:38 WARN SparkContext: Killing executors is not supported by current scheduler.
25/03/20 05:19:44 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:56)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:310)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:124)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$

In [ ]:
# Read the text file
lines = spark.read.text("Airline-Full-Non-Ag-DFE-Sentiment.csv")

# Split lines into parts
parts = lines.rdd.map(lambda line: line.value.split(","))

# Filter for valid parts
validParts = parts.filter(lambda line: len(line) >= 22 and line[14] is not None and line[21] is not None)

# Filter for negative sentiments
negatives = validParts.filter(lambda tweet: tweet[14] == "negative")

# Map tweets to a key-value pair (using tweet[10] as key)
rdd3 = negatives.map(lambda tweet: (tweet[10], 1))  # Added the missing closing parenthesis here

# Reduce by key to count occurrences
rdd4 = rdd3.reduceByKey(lambda x, y: x + y)

# Sort by the count in descending order
rdd5 = rdd4.sortBy(lambda x: x[1], ascending=False)

# Print the top 3 results
print(rdd5.take(3))


[('IND', 5092), ('PHL', 2202), ('ESP', 2134)]


25/03/18 16:50:12 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 355303 ms exceeds timeout 120000 ms
25/03/18 16:50:12 WARN SparkContext: Killing executors is not supported by current scheduler.


In [4]:
# Import required libraries
import pandas as pd
import plotly.express as px

# ---------------------------
# 1) Load the Data
# ---------------------------
# File path to your ratingvalue.py data
file_path = "anything.csv"  # Update if needed

# Read the CSV file
df = pd.read_csv(file_path)

# ---------------------------
# 2) Data Preprocessing
# ---------------------------
# Convert 'time_period' to datetime format
df["time_period"] = pd.to_datetime(df["time_period"], format="%Y-%m")

# Sort data by time_period
df = df.sort_values(by="time_period")

# ---------------------------
# 3) Apply Rolling Average for Smoothing
# ---------------------------
# Apply a rolling average over a 12-month window
df["rolling_avg"] = df["average_rating"].rolling(window=12, min_periods=1).mean()

# ---------------------------
# 4) Create Plotly Line Graph with Rolling Average
# ---------------------------
fig = px.line(
    df,
    x="time_period",
    y="rolling_avg",
    title="Smoothed Average Rating Over Time (12-Month Rolling Average)",
    labels={"time_period": "Time Period", "rolling_avg": "Average Rating"},
)

# ---------------------------
# 5) Customize Layout
# ---------------------------
fig.update_layout(
    xaxis_title="Time Period",
    yaxis_title="Average Rating",
    xaxis=dict(
        showgrid=True,
        tickangle=45,
        tickformat="%b %Y",  # Show Month-Year
    ),
    yaxis=dict(
        showgrid=True,
        tickformat=".2f",  # Format y-axis with 2 decimal places
    ),
    height=500,
    width=800,
    margin=dict(l=50, r=50, t=50, b=50),
)

# ---------------------------
# 6) Show the Plot
# ---------------------------
fig.show()


In [8]:
import pandas as pd
import plotly.express as px

# ---------------------------
# 1) Load and Preprocess Data
# ---------------------------
# File path to your ratingvalue.py data
file_path = "anything.csv"  # Update this if needed

# Read the CSV file
df = pd.read_csv(file_path)

# Convert 'time_period' to datetime format
df["time_period"] = pd.to_datetime(df["time_period"], format="%Y-%m")

# Create a 'year_month' column as period and then convert to string
df["year_month"] = df["time_period"].dt.to_period("M").astype(str)

# ---------------------------
# 2) Create Plotly Box Plot
# ---------------------------
fig_box = px.box(
    df,
    x="year_month",
    y="average_rating",
    title="Distribution of Ratings Over Time (Monthly)",
    labels={"year_month": "Month-Year", "average_rating": "Average Rating"},
)

# ---------------------------
# 3) Customize Layout
# ---------------------------
fig_box.update_layout(
    xaxis_title="Month-Year",
    yaxis_title="Average Rating",
    xaxis=dict(
        showgrid=True,
        tickangle=45,
        tickmode="array",
        tickvals=df["year_month"].iloc[::12],  # Show every 12 months to reduce clutter
    ),
    height=500,
    width=800,
    margin=dict(l=50, r=50, t=50, b=50),
)

# ---------------------------
# 4) Show the Plot
# ---------------------------
fig_box.show()


In [6]:
import plotly.graph_objects as go
import numpy as np

# Create pivot table with years as y-axis and months as x-axis
df["year"] = df["time_period"].dt.year
df["month"] = df["time_period"].dt.month
heatmap_data = df.pivot_table(
    values="average_rating",
    index="year",
    columns="month",
    aggfunc="mean"
)

# Create heatmap
fig_heatmap = go.Figure(
    data=go.Heatmap(
        z=heatmap_data.values,
        x=heatmap_data.columns,
        y=heatmap_data.index,
        colorscale="YlGnBu",
        hoverongaps=False,
    )
)

fig_heatmap.update_layout(
    title="Average Rating Heatmap by Month and Year",
    xaxis_title="Month",
    yaxis_title="Year",
    height=500,
    width=800,
    margin=dict(l=50, r=50, t=50, b=50),
)
fig_heatmap.show()


In [9]:
import pandas as pd

# Load the CSV file
df = pd.read_csv("anything.csv")

# Print basic information
print(f"CSV shape: {df.shape}")
print(f"CSV columns: {df.columns.tolist()}")

# Check for unique categories
if 'main_category' in df.columns:
    print(f"Unique categories: {df['main_category'].unique().tolist()}")

# Print a sample of the data
print("\nSample data:")
print(df.head())

CSV shape: (2623, 6)
CSV columns: ['main_category', 'time_period', 'average_rating', 'rating_count', 'month', 'year']
Unique categories: ['All Electronics', 'Cell Phone & Camera w. Accessories', 'Computers', 'Daily Gadgets', 'Media', 'Others', 'Software', 'Sports & Health', 'Toys & Games', 'Video Games']

Sample data:
     main_category time_period  average_rating  rating_count month  year
0  All Electronics     1999-11             3.0             1   Nov  1999
1  All Electronics     1999-12             3.0             4   Dec  1999
2  All Electronics     2000-02             5.0             1   Feb  2000
3  All Electronics     2000-03             5.0             2   Mar  2000
4  All Electronics     2000-04             5.0             1   Apr  2000
